# 26. Colormap comparison

The same anomaly map looks very different under different colormaps. This notebook shows one fixed-range map in several colormaps, side by side, for a good tile and defect tiles (README section 6).

**A colormap only changes how the map looks. It never changes the decision.** Detection uses the anomaly values and the threshold; the colours are drawn afterwards. Two colormaps on the same map give the same OK / DAMAGED result.

Display range: `vmax = VMAX_FACTOR x` the pixel threshold of the tile, so the middle of the colormap means "would be flagged" and the picture agrees with the decision. It is never per-image min-max, so a good tile does not show fake hot spots.

Output: a grid per example tile, and an overlay on the original tile (`cv2.addWeighted`). Pictures are saved to `results/colormap/`.

In [ ]:
import sys, os, importlib
import numpy as np, cv2
import matplotlib.pyplot as plt

sys.path.insert(0, "../src/methods")
import filters as F
import evaluate as E
importlib.reload(F); importlib.reload(E)

# ---- settings ---------------------------------------------------------------
FABRIC = "02"                         # fabric code
METHOD_NAME = "background_subtraction"  # any tile-only method in F.METHODS
VMAX_FACTOR = 2.0                     # display range = VMAX_FACTOR x pixel threshold (2 = threshold sits at the middle of the colormap)
OUT_DIR = "../results/colormap"
SETTINGS = dict(thr_k=4.0, min_area=4, score_k=25, score_sigma=2.0, normalize=True)   # same as every experiment

# colormaps to compare: (name, OpenCV constant, matplotlib name)
COLORMAPS = [("JET", cv2.COLORMAP_JET, "jet"),
             ("TURBO", cv2.COLORMAP_TURBO, "turbo"),
             ("INFERNO", cv2.COLORMAP_INFERNO, "inferno"),
             ("MAGMA", cv2.COLORMAP_MAGMA, "magma"),
             ("VIRIDIS", cv2.COLORMAP_VIRIDIS, "viridis"),
             ("HOT", cv2.COLORMAP_HOT, "hot")]
os.makedirs(OUT_DIR, exist_ok=True)
method = F.METHODS[METHOD_NAME]

## 1. Threshold and example tiles
The threshold comes from the good_cal strips of the fabric (README section 7). Examples: the largest defect, the smallest defect and a good tile.

In [ ]:
res = E.calibrate(method=method, fabric=FABRIC, **SETTINGS)
assert res is not None, f"not enough tiles for fabric {FABRIC}"
examples = E.example_tiles(FABRIC)
for name, p, area in examples:
    print(f"{name}: {p.name}" + (f", {area} px" if area else ""))
print(f"threshold (score) = {res['thr']:.3f}")

## 2. Side by side
Each row is one tile: the tile, then its anomaly map in every colormap with the same fixed range. The title of each row says what the decision is, and it is the same for every colormap.

In [ ]:
fig, axes = plt.subplots(len(examples), 1 + len(COLORMAPS), figsize=(3 * (1 + len(COLORMAPS)), 3.2 * len(examples)))
for r, (name, p, area) in enumerate(examples):
    tile = E.load(p)
    amap = method(tile)
    score, flagged, pix_thr, _, _ = E.judge(amap, res)
    shown = F.display(amap, VMAX_FACTOR * pix_thr)               # [0, 1], fixed range
    axes[r, 0].imshow(tile, cmap="gray", vmin=0, vmax=255)
    axes[r, 0].set_title(f"{name}\n{'FLAGGED' if flagged else 'not flagged'} (score {score:.2f})", fontsize=10)
    for c, (cname, _, mname) in enumerate(COLORMAPS, 1):
        axes[r, c].imshow(shown, cmap=mname, vmin=0, vmax=1)
        axes[r, c].set_title(cname, fontsize=10)
    for a in axes[r]:
        a.axis("off")
fig.suptitle(f"{METHOD_NAME}, fabric {FABRIC}: same map, same range, different colormaps", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.97), h_pad=2.5)
fig.savefig(os.path.join(OUT_DIR, f"grid_{METHOD_NAME}_fabric{FABRIC}.png"), dpi=110)
plt.show()

## 3. Overlay on the original
The map drawn over the tile with `cv2.addWeighted` (45 % tile, 55 % colour), the way a heat map is usually shown on a photo. Only the largest defect and the good tile are shown.

In [ ]:
def overlay(tile, shown, cv_cmap, alpha=0.55):
    colour = cv2.applyColorMap(np.uint8(np.clip(shown, 0, 1) * 255), cv_cmap)
    return cv2.addWeighted(cv2.cvtColor(tile, cv2.COLOR_GRAY2BGR), 1 - alpha, colour, alpha, 0)

pick = [examples[0], examples[-1]]       # largest defect and good tile
fig, axes = plt.subplots(len(pick), len(COLORMAPS), figsize=(3 * len(COLORMAPS), 3.2 * len(pick)))
for r, (name, p, area) in enumerate(pick):
    tile = E.load(p)
    amap = method(tile)
    _, _, pix_thr, _, _ = E.judge(amap, res)
    shown = F.display(amap, VMAX_FACTOR * pix_thr)
    for c, (cname, cv_cmap, _) in enumerate(COLORMAPS):
        axes[r, c].imshow(cv2.cvtColor(overlay(tile, shown, cv_cmap), cv2.COLOR_BGR2RGB))
        axes[r, c].set_title(f"{cname} ({name})" if r == 0 else f"{cname} ({name})", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"overlay_{METHOD_NAME}_fabric{FABRIC}.png"), dpi=110)
plt.show()

## 4. Colormap does not change the decision
The score and the OK / DAMAGED result are computed from the anomaly values before any colour is applied. The check below makes that explicit: the damage mask (pixels above the threshold) is identical whichever colormap is used to draw it.

In [ ]:
name, p, area = examples[0]
tile = E.load(p)
amap = method(tile)
score, flagged, pix_thr, _, _ = E.judge(amap, res)
mask = E.damage_mask(amap, res, pix_thr)
print(f"{name}: score {score:.2f}, threshold {res['thr']:.2f} -> {'DAMAGED' if flagged else 'OK'}; damage mask = {int(mask.sum())} px")
print("The same numbers are used for every colormap; only the colours differ.")

## 5. Which colormap to use

| Colormap | Notes |
|---|---|
| JET | Familiar, but adds fake bands and edges |
| TURBO | Smoother replacement for JET |
| INFERNO / MAGMA / VIRIDIS | Perceptually smooth, honest brightness; good for reports |
| HOT | Simple black-red-white, shows peaks |

The notebooks and the app use INFERNO. Change `COLORMAPS` above to try others (any `cv2.COLORMAP_*` with its matplotlib name).